#PARCIAL 2 ANALISIS DEL MERCADO LABORAL COLOMBIANO JUNIO 2026

##integrantes: johan velez,jesus cossio 

## Objetivo general

Analizar las principales características del mercado laboral colombiano a junio de 2026 utilizando los microdatos de la Gran Encuesta Integrada de Hogares (GEIH) del DANE.

## Objetivos específicos

1. Calcular los principales indicadores del mercado laboral para el total nacional.
2. Comparar las tasas de desempleo de hombres y mujeres entre los departamentos de Colombia.
3. Identificar los grupos de edad con mayores y menores tasas de desempleo por departamento.
4. Determinar los tres departamentos con menor tasa de desempleo de mujeres entre los 18 y los 25 años.
5. Calcular el promedio de ingresos laborales de los residentes de Risaralda, Caldas, Quindío, Antioquia y Valle del Cauca.

## Fuente de información y metodología

Se utilizarán los microdatos de la GEIH publicados por el Departamento Administrativo Nacional de Estadística (DANE), correspondientes a junio de 2026. Los datos se prepararán y analizarán en Databricks. Los cálculos deberán considerar las definiciones oficiales de los indicadores laborales y los factores de expansión pertinentes para obtener estimaciones representativas.

Los resultados se presentarán mediante tablas y se acompañarán de interpretaciones que expliquen su significado económico.

In [0]:
ruta = "/Volumes/workspace/electiva/geih_2026/Características generales, seguridad social en salud y educación.CSV"

caracteristicas = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(ruta)
)

display(caracteristicas.limit(10))

In [0]:
caracteristicas.printSchema()
print("Total de registros:", caracteristicas.count())

In [0]:
archivos = dbutils.fs.ls("/Volumes/workspace/electiva/geih_2026/")

ruta_ft = next(
    archivo.path
    for archivo in archivos
    if "Fuerza" in archivo.name and "trabajo" in archivo.name
)

print("Ruta encontrada:", repr(ruta_ft))

fuerza_trabajo = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(ruta_ft)
)

fuerza_trabajo.printSchema()
print("Total de registros:", fuerza_trabajo.count())

display(fuerza_trabajo.limit(10))

In [0]:
from pyspark.sql import functions as F

print("Distribución de FT:")
display(
    fuerza_trabajo
    .groupBy("FT")
    .agg(
        F.count("*").alias("registros"),
        F.sum("FEX_C18").alias("suma_expandida")
    )
    .orderBy("FT")
)

print("Distribución de FFT:")
display(
    fuerza_trabajo
    .groupBy("FFT")
    .agg(
        F.count("*").alias("registros"),
        F.sum("FEX_C18").alias("suma_expandida")
    )
    .orderBy("FFT")
)

print("Distribución de PET:")
display(
    fuerza_trabajo
    .groupBy("PET")
    .agg(
        F.count("*").alias("registros"),
        F.sum("FEX_C18").alias("suma_expandida")
    )
    .orderBy("PET")
)

In [0]:
archivos = dbutils.fs.ls("/Volumes/workspace/electiva/geih_2026/")

ruta_ocupados = next(
    archivo.path
    for archivo in archivos
    if archivo.name.startswith("Ocupados")
)

ocupados = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(ruta_ocupados)
)

ocupados.printSchema()
print("Total de registros:", ocupados.count())

display(ocupados.limit(10))

In [0]:
from pyspark.sql import functions as F

print("Resumen de INGLABO:")
display(
    ocupados.select("INGLABO").summary()
)

print("Registros con ingreso laboral:")
display(
    ocupados.select(
        F.count("*").alias("total_registros"),
        F.count("INGLABO").alias("ingresos_no_nulos"),
        F.sum(F.when(F.col("INGLABO").isNull(), 1).otherwise(0))
         .alias("ingresos_nulos"),
        F.min("INGLABO").alias("ingreso_minimo"),
        F.max("INGLABO").alias("ingreso_maximo"),
        F.avg("INGLABO").alias("promedio_simple")
    )
)

In [0]:

from pyspark.sql import functions as F

# Departamentos solicitados:
# 17 = Caldas, 63 = Quindío, 66 = Risaralda,
# 05 = Antioquia, 76 = Valle del Cauca

departamentos = {
    17: "Caldas",
    63: "Quindío",
    66: "Risaralda",
    5: "Antioquia",
    76: "Valle del Cauca"
}

# Promedio ponderado de ingresos laborales
promedios = (
    ocupados
    .filter(
        F.col("DPTO").isin(list(departamentos.keys()))
        & F.col("INGLABO").isNotNull()
        & F.col("FEX_C18").isNotNull()
        & (F.col("FEX_C18") > 0)
    )
    .groupBy("DPTO")
    .agg(
        F.count("*").alias("registros"),
        (
            F.sum(F.col("INGLABO") * F.col("FEX_C18"))
            / F.sum("FEX_C18")
        ).alias("ingreso_promedio_ponderado")
    )
    .orderBy("DPTO")
)

display(promedios)


In [0]:

from pyspark.sql import functions as F

print("Valores de PET:")
display(
    fuerza_trabajo
    .groupBy("PET")
    .agg(
        F.count("*").alias("registros"),
        F.sum("FEX_C18").alias("poblacion_expandida")
    )
    .orderBy("PET")
)

print("Valores de FT:")
display(
    fuerza_trabajo
    .groupBy("FT")
    .agg(
        F.count("*").alias("registros"),
        F.sum("FEX_C18").alias("poblacion_expandida")
    )
    .orderBy("FT")
)

print("Valores de FFT:")
display(
    fuerza_trabajo
    .groupBy("FFT")
    .agg(
        F.count("*").alias("registros"),
        F.sum("FEX_C18").alias("poblacion_expandida")
    )
    .orderBy("FFT")
)


In [0]:

from pyspark.sql import functions as F

print("Distribución de P6240:")
display(
    fuerza_trabajo
    .groupBy("P6240")
    .agg(
        F.count("*").alias("registros"),
        F.sum("FEX_C18").alias("poblacion_expandida")
    )
    .orderBy("P6240")
)

print("Columnas relacionadas con clasificación laboral:")
print([
    c for c in fuerza_trabajo.columns
    if any(p in c.upper() for p in ["OCUP", "DESOC", "INACT", "PEA", "PET", "FT", "FFT"])
])


In [0]:

print("Variables del módulo Fuerza de trabajo:")
print(fuerza_trabajo.columns)


In [0]:

archivos = dbutils.fs.ls("/Volumes/workspace/electiva/geih_2026/")

ruta_no_ocupados = next(
    archivo.path
    for archivo in archivos
    if archivo.name == "No ocupados.CSV"
)

no_ocupados = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(ruta_no_ocupados)
)

print("Total de registros:", no_ocupados.count())
print("Variables disponibles:")
print(no_ocupados.columns)


In [0]:

from pyspark.sql import functions as F

print("Distribución de DSI en No ocupados:")

display(
    no_ocupados
    .groupBy("DSI")
    .agg(
        F.count("*").alias("registros"),
        F.sum("FEX_C18").alias("poblacion_expandida")
    )
    .orderBy("DSI")
)

print("Distribución de FFT:")

display(
    no_ocupados
    .groupBy("FFT")
    .agg(
        F.count("*").alias("registros"),
        F.sum("FEX_C18").alias("poblacion_expandida")
    )
    .orderBy("FFT")
)


In [0]:

from pyspark.sql import functions as F

llaves = ["PERIODO", "DIRECTORIO", "SECUENCIA_P", "HOGAR", "ORDEN"]

for nombre, df in [
    ("Características", caracteristicas),
    ("Fuerza de trabajo", fuerza_trabajo),
    ("Ocupados", ocupados),
    ("No ocupados", no_ocupados)
]:
    print(f"\n{nombre}")
    display(
        df.agg(
            F.count("*").alias("registros"),
            F.countDistinct(*llaves).alias("personas_distintas")
        )
    )


In [0]:

from pyspark.sql import functions as F

llaves = ["PERIODO", "DIRECTORIO", "SECUENCIA_P", "HOGAR", "ORDEN"]

ocupados_ids = ocupados.select(*llaves).distinct()
no_ocupados_ids = no_ocupados.select(*llaves).distinct()

print("Personas que aparecen en ambos módulos:")
display(
    ocupados_ids.join(no_ocupados_ids, on=llaves, how="inner")
    .count()
)

print("Personas de Ocupados que aparecen en Fuerza de trabajo:")
display(
    ocupados_ids.join(
        fuerza_trabajo.select(*llaves).distinct(),
        on=llaves,
        how="inner"
    ).count()
)

print("Personas de No ocupados que aparecen en Fuerza de trabajo:")
display(
    no_ocupados_ids.join(
        fuerza_trabajo.select(*llaves).distinct(),
        on=llaves,
        how="inner"
    ).count()
)


In [0]:

from pyspark.sql import functions as F

display(
    no_ocupados
    .join(
        fuerza_trabajo.select(
            "PERIODO", "DIRECTORIO", "SECUENCIA_P",
            "HOGAR", "ORDEN", "P6240"
        ),
        on=["PERIODO", "DIRECTORIO", "SECUENCIA_P", "HOGAR", "ORDEN"],
        how="left"
    )
    .groupBy("DSI", "P6240")
    .agg(
        F.count("*").alias("registros"),
        F.sum("FEX_C18").alias("poblacion_expandida")
    )
    .orderBy("DSI", "P6240")
)


In [0]:

from pyspark.sql import functions as F

# Población económicamente activa (PEA)
pea = (
    fuerza_trabajo
    .filter(F.col("FT") == 1)
    .agg(F.sum("FEX_C18").alias("total"))
    .first()["total"]
)

# Población desocupada (DSI = 1)
desocupados_total = (
    no_ocupados
    .filter(F.col("DSI") == 1)
    .agg(F.sum("FEX_C18").alias("total"))
    .first()["total"]
)

# Población en edad de trabajar (PET)
pet = (
    fuerza_trabajo
    .filter(F.col("PET") == 1)
    .agg(F.sum("FEX_C18").alias("total"))
    .first()["total"]
)

# Población fuera de la fuerza de trabajo (PEI)
pei = (
    fuerza_trabajo
    .filter(F.col("FFT") == 1)
    .agg(F.sum("FEX_C18").alias("total"))
    .first()["total"]
)

ocupados_total = pea - desocupados_total

resultados = [
    ("PET", pet),
    ("PEA", pea),
    ("PEI", pei),
    ("Desocupados", desocupados_total),
    ("Ocupados estimados", ocupados_total),
    ("Tasa de desempleo (%)", desocupados_total / pea * 100),
    ("Tasa de ocupación (%)", ocupados_total / pet * 100)
]

display(
    spark.createDataFrame(
        resultados, ["Indicador", "Resultado"]
    )
)


In [0]:

from pyspark.sql import functions as F

print("Distribución de sexo (P3271):")
display(
    caracteristicas
    .groupBy("P3271")
    .agg(
        F.count("*").alias("registros"),
        F.sum("FEX_C18").alias("poblacion_expandida")
    )
    .orderBy("P3271")
)

print("Departamentos disponibles (DPTO):")
display(
    caracteristicas
    .groupBy("DPTO")
    .agg(F.count("*").alias("registros"))
    .orderBy("DPTO")
)


In [0]:

from pyspark.sql import functions as F

llaves = ["PERIODO", "DIRECTORIO", "SECUENCIA_P", "HOGAR", "ORDEN"]

# Datos personales: sexo y departamento
datos_persona = caracteristicas.select(
    *llaves, "P3271", "DPTO"
)

# PEA por sexo y departamento
pea_grupo = (
    fuerza_trabajo
    .filter(F.col("FT") == 1)
    .join(datos_persona, on=llaves, how="inner")
    .groupBy("DPTO", "P3271")
    .agg(F.sum("FEX_C18").alias("PEA"))
)

# Personas desocupadas por sexo y departamento
desocupados_grupo = (
    no_ocupados
    .filter(F.col("DSI") == 1)
    .join(datos_persona, on=llaves, how="inner")
    .groupBy("DPTO", "P3271")
    .agg(F.sum("FEX_C18").alias("Desocupados"))
)

# Calcular tasa de desempleo
tasas = (
    pea_grupo
    .join(desocupados_grupo, on=["DPTO", "P3271"], how="left")
    .fillna({"Desocupados": 0})
    .withColumn(
        "Tasa_desempleo_pct",
        F.col("Desocupados") / F.col("PEA") * 100
    )
    .withColumn(
        "Sexo",
        F.when(F.col("P3271") == 1, "Hombres")
         .when(F.col("P3271") == 2, "Mujeres")
         .otherwise("Sin clasificar")
    )
    .select(
        "DPTO", "Sexo", "PEA", "Desocupados",
        F.round("Tasa_desempleo_pct", 2).alias("Tasa_desempleo_pct")
    )
    .orderBy("DPTO", "Sexo")
)

display(tasas)


In [0]:

from pyspark.sql import functions as F

llaves = ["PERIODO", "DIRECTORIO", "SECUENCIA_P", "HOGAR", "ORDEN"]

# Datos personales: una sola columna DPTO y una sola de sexo
datos_persona = caracteristicas.select(
    *llaves,
    F.col("DPTO").alias("departamento"),
    F.col("P3271").alias("sexo")
)

# PEA por departamento y sexo
pea_grupo = (
    fuerza_trabajo
    .filter(F.col("FT") == 1)
    .select(*llaves, "FEX_C18")
    .join(datos_persona, on=llaves, how="inner")
    .groupBy("departamento", "sexo")
    .agg(F.sum("FEX_C18").alias("PEA"))
)

# Desocupados por departamento y sexo
desocupados_grupo = (
    no_ocupados
    .filter(F.col("DSI") == 1)
    .select(*llaves, "FEX_C18")
    .join(datos_persona, on=llaves, how="inner")
    .groupBy("departamento", "sexo")
    .agg(F.sum("FEX_C18").alias("Desocupados"))
)

# Tasa de desempleo
tasas = (
    pea_grupo
    .join(
        desocupados_grupo,
        on=["departamento", "sexo"],
        how="left"
    )
    .fillna({"Desocupados": 0})
    .withColumn(
        "Tasa_desempleo_pct",
        F.col("Desocupados") / F.col("PEA") * 100
    )
    .withColumn(
        "Sexo",
        F.when(F.col("sexo") == 1, "Hombres")
         .when(F.col("sexo") == 2, "Mujeres")
         .otherwise("Sin clasificar")
    )
    .select(
        F.col("departamento").alias("DPTO"),
        "Sexo",
        F.round("PEA", 0).alias("PEA"),
        F.round("Desocupados", 0).alias("Desocupados"),
        F.round("Tasa_desempleo_pct", 2).alias("Tasa_desempleo_pct")
    )
    .orderBy("DPTO", "Sexo")
)

display(tasas)


In [0]:

from pyspark.sql import functions as F

llaves = ["PERIODO", "DIRECTORIO", "SECUENCIA_P", "HOGAR", "ORDEN"]

# Edad y departamento de cada persona
datos_edad = caracteristicas.select(
    *llaves,
    F.col("DPTO").alias("departamento"),
    F.col("P6040").cast("int").alias("edad")
).withColumn(
    "grupo_edad",
    F.when((F.col("edad") >= 18) & (F.col("edad") < 24), "18-23")
     .when((F.col("edad") >= 24) & (F.col("edad") < 30), "24-29")
     .when((F.col("edad") >= 30) & (F.col("edad") < 40), "30-39")
     .when((F.col("edad") >= 40) & (F.col("edad") < 50), "40-49")
     .when(F.col("edad") >= 60, "60+")
)

# PEA por departamento y grupo de edad
pea_edad = (
    fuerza_trabajo
    .filter((F.col("FT") == 1))
    .select(*llaves, "FEX_C18")
    .join(datos_edad, on=llaves, how="inner")
    .filter(F.col("grupo_edad").isNotNull())
    .groupBy("departamento", "grupo_edad")
    .agg(F.sum("FEX_C18").alias("PEA"))
)

# Desocupados por departamento y grupo de edad
desocupados_edad = (
    no_ocupados
    .filter(F.col("DSI") == 1)
    .select(*llaves, "FEX_C18")
    .join(datos_edad, on=llaves, how="inner")
    .filter(F.col("grupo_edad").isNotNull())
    .groupBy("departamento", "grupo_edad")
    .agg(F.sum("FEX_C18").alias("Desocupados"))
)

# Tasa de desempleo
tasas_edad = (
    pea_edad
    .join(
        desocupados_edad,
        on=["departamento", "grupo_edad"],
        how="left"
    )
    .fillna({"Desocupados": 0})
    .withColumn(
        "Tasa_desempleo_pct",
        F.col("Desocupados") / F.col("PEA") * 100
    )
    .select(
        F.col("departamento").alias("DPTO"),
        "grupo_edad",
        F.round("PEA", 0).alias("PEA"),
        F.round("Desocupados", 0).alias("Desocupados"),
        F.round("Tasa_desempleo_pct", 2).alias("Tasa_desempleo_pct")
    )
    .orderBy("DPTO", "grupo_edad")
)

display(tasas_edad)


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Orden de los grupos de edad
orden_edad = (
    F.when(F.col("grupo_edad") == "18-23", 1)
     .when(F.col("grupo_edad") == "24-29", 2)
     .when(F.col("grupo_edad") == "30-39", 3)
     .when(F.col("grupo_edad") == "40-49", 4)
     .when(F.col("grupo_edad") == "60+", 5)
)

base = tasas_edad.withColumn("orden_edad", orden_edad)

ventana = Window.partitionBy("DPTO")

resumen_edades = (
    base
    .withColumn("tasa_maxima", F.max("Tasa_desempleo_pct").over(ventana))
    .withColumn("tasa_minima", F.min("Tasa_desempleo_pct").over(ventana))
    .groupBy("DPTO")
    .agg(
        F.max(
            F.when(
                F.col("Tasa_desempleo_pct") == F.col("tasa_maxima"),
                F.concat(
                    F.col("grupo_edad"),
                    F.lit(" ("),
                    F.col("Tasa_desempleo_pct").cast("string"),
                    F.lit("%)")
                )
            )
        ).alias("Grupo_con_tasa_mas_alta"),
        F.max(
            F.when(
                F.col("Tasa_desempleo_pct") == F.col("tasa_minima"),
                F.concat(
                    F.col("grupo_edad"),
                    F.lit(" ("),
                    F.col("Tasa_desempleo_pct").cast("string"),
                    F.lit("%)")
                )
            )
        ).alias("Grupo_con_tasa_mas_baja")
    )
    .orderBy("DPTO")
)

display(resumen_edades)


In [0]:
from pyspark.sql import functions as F

llaves = ["PERIODO", "DIRECTORIO", "SECUENCIA_P", "HOGAR", "ORDEN"]

# Mujeres de 18 a 25 años, ambos extremos incluidos
mujeres_18_25 = (
    caracteristicas
    .filter(
        (F.col("P3271") == 2) &
        (F.col("P6040").between(18, 25))
    )
    .select(*llaves, F.col("DPTO").alias("departamento"))
)

# Población económicamente activa
pea_mujeres = (
    fuerza_trabajo
    .filter(F.col("FT") == 1)
    .select(*llaves, "FEX_C18")
    .join(mujeres_18_25, on=llaves, how="inner")
    .groupBy("departamento")
    .agg(F.sum("FEX_C18").alias("PEA"))
)

# Mujeres desempleadas
desocupadas_mujeres = (
    no_ocupados
    .filter(F.col("DSI") == 1)
    .select(*llaves, "FEX_C18")
    .join(mujeres_18_25, on=llaves, how="inner")
    .groupBy("departamento")
    .agg(F.sum("FEX_C18").alias("Desocupadas"))
)

# Tasa de desempleo y los tres departamentos con menor tasa
tasas_mujeres = (
    pea_mujeres
    .join(desocupadas_mujeres, on="departamento", how="left")
    .fillna({"Desocupadas": 0})
    .withColumn(
        "Tasa_desempleo_pct",
        F.col("Desocupadas") / F.col("PEA") * 100
    )
    .orderBy("Tasa_desempleo_pct")
)

display(tasas_mujeres.limit(3))

In [0]:
from pyspark.sql import functions as F

# Códigos oficiales de los departamentos
departamentos = {
    66: "Risaralda",
    17: "Caldas",
    63: "Quindío",
    5: "Antioquia",
    76: "Valle del Cauca"
}

# Calcular promedio ponderado de INGLABO
promedio_ingresos = (
    ocupados
    .filter(
        F.col("DPTO").isin(list(departamentos.keys())) &
        F.col("INGLABO").isNotNull()
    )
    .groupBy("DPTO")
    .agg(
        F.sum(F.col("INGLABO") * F.col("FEX_C18")).alias("Ingreso_ponderado"),
        F.sum("FEX_C18").alias("Poblacion_ponderada"),
        F.count("INGLABO").alias("Registros_muestra")
    )
    .withColumn(
        "Promedio_ingreso",
        F.col("Ingreso_ponderado") / F.col("Poblacion_ponderada")
    )
    .orderBy("DPTO")
)

# Mostrar resultados en pesos colombianos
display(
    promedio_ingresos.select(
        F.col("DPTO").alias("Codigo_departamento"),
        F.round("Promedio_ingreso", 0).alias("Promedio_ingreso_COP"),
        "Registros_muestra"
    )
)

In [0]:

from pyspark.sql import functions as F

llaves = ["PERIODO", "DIRECTORIO", "SECUENCIA_P", "HOGAR", "ORDEN"]

# Datos de sexo y departamento
datos_persona = (
    caracteristicas
    .select(
        *llaves,
        F.col("DPTO").alias("departamento"),
        F.col("P3271").alias("sexo")
    )
)

# Población económicamente activa por sexo y departamento
pea_sexo = (
    fuerza_trabajo
    .filter(F.col("FT") == 1)
    .select(*llaves, "FEX_C18")
    .join(datos_persona, on=llaves, how="inner")
    .groupBy("departamento", "sexo")
    .agg(F.sum("FEX_C18").alias("PEA"))
)

# Personas desocupadas por sexo y departamento
desocupados_sexo = (
    no_ocupados
    .filter(F.col("DSI") == 1)
    .select(*llaves, "FEX_C18")
    .join(datos_persona, on=llaves, how="inner")
    .groupBy("departamento", "sexo")
    .agg(F.sum("FEX_C18").alias("Desocupados"))
)

# Calcular la tasa de desempleo
tasas_sexo = (
    pea_sexo
    .join(
        desocupados_sexo,
        on=["departamento", "sexo"],
        how="left"
    )
    .fillna({"Desocupados": 0})
    .withColumn(
        "Tasa_desempleo_pct",
        F.col("Desocupados") / F.col("PEA") * 100
    )
    .withColumnRenamed("departamento", "DPTO")
    .withColumn(
        "Sexo",
        F.when(F.col("sexo") == 1, "Hombres")
         .when(F.col("sexo") == 2, "Mujeres")
         .otherwise("Sin clasificar")
    )
)

display(tasas_sexo.orderBy("DPTO", "sexo"))


In [0]:

from pyspark.sql import functions as F

nombres_dpto = {
    5: "Antioquia", 8: "Atlántico", 11: "Bogotá D.C.",
    13: "Bolívar", 15: "Boyacá", 17: "Caldas",
    18: "Caquetá", 19: "Cauca", 20: "Cesar",
    23: "Córdoba", 25: "Cundinamarca", 27: "Chocó",
    41: "Huila", 44: "La Guajira", 47: "Magdalena",
    50: "Meta", 52: "Nariño", 54: "Norte de Santander",
    63: "Quindío", 66: "Risaralda", 68: "Santander",
    70: "Sucre", 73: "Tolima", 76: "Valle del Cauca",
    81: "Arauca", 85: "Casanare", 86: "Putumayo",
    88: "San Andrés", 91: "Amazonas", 94: "Guainía",
    95: "Guaviare", 97: "Vaupés", 99: "Vichada"
}

poblacion_total = caracteristicas.agg(
    F.sum("FEX_C18").alias("total")
).first()["total"]

pet = fuerza_trabajo.filter(
    F.col("PET") == 1
).agg(F.sum("FEX_C18")).first()[0]

pea = fuerza_trabajo.filter(
    F.col("FT") == 1
).agg(F.sum("FEX_C18")).first()[0]

pei = fuerza_trabajo.filter(
    F.col("FFT") == 1
).agg(F.sum("FEX_C18")).first()[0]

desocupados = no_ocupados.filter(
    F.col("DSI") == 1
).agg(F.sum("FEX_C18")).first()[0]

ocupados_nacionales = pea - desocupados
poblacion_fuera_pet = poblacion_total - pet

tasa_desempleo = desocupados / pea * 100
tasa_ocupacion = ocupados_nacionales / pet * 100

resultados_sexo = tasas_sexo.collect()

hombres = [
    r for r in resultados_sexo if r["Sexo"] == "Hombres"
]
mujeres = [
    r for r in resultados_sexo if r["Sexo"] == "Mujeres"
]

h_mayor = max(hombres, key=lambda r: r["Tasa_desempleo_pct"])
h_menor = min(hombres, key=lambda r: r["Tasa_desempleo_pct"])
m_mayor = max(mujeres, key=lambda r: r["Tasa_desempleo_pct"])
m_menor = min(mujeres, key=lambda r: r["Tasa_desempleo_pct"])

def nombre_tasa(fila):
    departamento = nombres_dpto.get(int(fila["DPTO"]), str(fila["DPTO"]))
    return f"{departamento}: {float(fila['Tasa_desempleo_pct']):.2f}%"

filas_edades = ""
for r in resumen_edades.collect():
    departamento = nombres_dpto.get(int(r["DPTO"]), str(r["DPTO"]))
    filas_edades += (
        f"<tr><td>{departamento}</td>"
        f"<td>{r['Grupo_con_tasa_mas_alta']}</td>"
        f"<td>{r['Grupo_con_tasa_mas_baja']}</td></tr>"
    )

tres_menores = tasas_mujeres.orderBy(
    "Tasa_desempleo_pct"
).limit(3).collect()

filas_mujeres = ""
for r in tres_menores:
    departamento = nombres_dpto.get(
        int(r["departamento"]), str(r["departamento"])
    )
    filas_mujeres += (
        f"<tr><td>{departamento}</td>"
        f"<td>{float(r['Tasa_desempleo_pct']):.2f}%</td></tr>"
    )

filas_ingresos = ""
for r in promedio_ingresos.collect():
    departamento = nombres_dpto.get(int(r["DPTO"]), str(r["DPTO"]))
    valor = r["Promedio_ingreso"]
    if valor is not None:
        filas_ingresos += (
            f"<tr><td>{departamento}</td>"
            f"<td>${valor:,.0f}</td></tr>"
        )

informe = f"""
<div style="font-family:Arial,sans-serif;line-height:1.65;color:#222">

<h1>ANÁLISIS DEL MERCADO LABORAL EN COLOMBIA</h1>
<h3>Gran Encuesta Integrada de Hogares (GEIH), junio de 2026</h3>

<p><b>Presentado por:</b> Johan Vélez y Jesús Cossio</p>
<p><b>Fuente de información:</b> Microdatos de la GEIH del DANE,
junio de 2026. Los cálculos utilizan el factor de expansión FEX_C18.</p>

<hr>

<h2>1. Indicadores nacionales del mercado laboral</h2>

<table border="1" cellpadding="7" cellspacing="0"
style="border-collapse:collapse;width:100%">
<tr><th align="left">Indicador</th><th align="right">Estimación</th></tr>
<tr><td>Población total estimada</td><td align="right">{poblacion_total:,.0f}</td></tr>
<tr><td>Población en edad de trabajar (PET)</td><td align="right">{pet:,.0f}</td></tr>
<tr><td>Población fuera de la PET, estimada por diferencia</td><td align="right">{poblacion_fuera_pet:,.0f}</td></tr>
<tr><td>Población económicamente activa (PEA)</td><td align="right">{pea:,.0f}</td></tr>
<tr><td>Población económicamente inactiva (PEI)</td><td align="right">{pei:,.0f}</td></tr>
<tr><td>Personas ocupadas estimadas</td><td align="right">{ocupados_nacionales:,.0f}</td></tr>
<tr><td>Personas desocupadas estimadas</td><td align="right">{desocupados:,.0f}</td></tr>
<tr><td>Tasa de desempleo</td><td align="right">{tasa_desempleo:.2f}%</td></tr>
<tr><td>Ocupados como proporción de la PET</td><td align="right">{tasa_ocupacion:.2f}%</td></tr>
</table>

<p><b>Interpretación:</b> La población económicamente activa comprende
a las personas ocupadas y a las desocupadas que participan en el mercado
laboral. La población económicamente inactiva corresponde a quienes
están fuera de la fuerza de trabajo. La tasa de desempleo expresa la
proporción de personas desocupadas dentro de la PEA.</p>

<h2>2. Tasa de desempleo por sexo y departamento</h2>

<table border="1" cellpadding="7" cellspacing="0"
style="border-collapse:collapse;width:100%">
<tr><th align="left">Sexo</th><th align="left">Tasa más alta</th>
<th align="left">Tasa más baja</th></tr>
<tr><td>Hombres</td><td>{nombre_tasa(h_mayor)}</td>
<td>{nombre_tasa(h_menor)}</td></tr>
<tr><td>Mujeres</td><td>{nombre_tasa(m_mayor)}</td>
<td>{nombre_tasa(m_menor)}</td></tr>
</table>

<p><b>Interpretación:</b> Las tasas de desempleo presentan diferencias
entre hombres y mujeres y entre departamentos. Estas variaciones
permiten identificar territorios con mayores dificultades de acceso
al empleo. Los resultados de departamentos con muestras pequeñas
deben interpretarse con precaución.</p>

<h2>3. Tasa de desempleo por grupo de edad</h2>

<table border="1" cellpadding="6" cellspacing="0"
style="border-collapse:collapse;width:100%">
<tr><th>Departamento</th><th>Grupo con tasa más alta</th>
<th>Grupo con tasa más baja</th></tr>
{filas_edades}
</table>

<p><b>Interpretación:</b> En la mayoría de los departamentos, el grupo
de 18 a 23 años registra la mayor tasa de desempleo, mientras que
el grupo de 60 años o más presenta la menor en la mayoría de los casos.
Este comportamiento sugiere mayores dificultades de inserción laboral
para la población joven. Una tasa de cero por ciento en la muestra
no significa necesariamente que el desempleo real sea inexistente.</p>

<h2>4. Departamentos con menor desempleo en mujeres de 18 a 25 años</h2>

<table border="1" cellpadding="7" cellspacing="0"
style="border-collapse:collapse;width:100%">
<tr><th>Departamento</th><th>Tasa de desempleo</th></tr>
{filas_mujeres}
</table>

<p><b>Interpretación:</b> La tabla presenta los tres departamentos con
menor tasa estimada de desempleo entre las mujeres de 18 a 25 años.
Las estimaciones deben evaluarse considerando el tamaño de la población
económicamente activa de cada departamento. En particular, una tasa
de cero por ciento basada en una muestra pequeña requiere cautela.</p>

<h2>5. Promedio de ingresos laborales por departamento</h2>

<table border="1" cellpadding="7" cellspacing="0"
style="border-collapse:collapse;width:100%">
<tr><th>Departamento</th><th>Ingreso laboral promedio (COP)</th></tr>
{filas_ingresos}
</table>

<p><b>Interpretación:</b> Entre los cinco departamentos analizados,
Antioquia presenta el mayor ingreso laboral promedio ponderado,
mientras que Valle del Cauca presenta el menor. Los valores se
calcularon a partir de la variable INGLABO, utilizando FEX_C18
y considerando los registros con ingresos no nulos.</p>

<h2>Conclusión general</h2>

<p>El análisis permite comparar indicadores del mercado laboral
colombiano por departamento, sexo y edad, además de observar las
diferencias en los ingresos laborales. Los resultados muestran
variaciones territoriales y diferencias entre grupos poblacionales.
Estas estimaciones constituyen una referencia para analizar la
situación laboral del país, teniendo en cuenta las definiciones
estadísticas y las limitaciones propias de una encuesta.</p>

<p><b>Nota metodológica:</b> La población fuera de la PET se presenta
como una estimación obtenida al restar la PET de la población total
expandida. Su interpretación depende de la comparabilidad de las
variables y de las definiciones del diccionario oficial de la GEIH.</p>

</div>
"""

displayHTML(informe)


In [0]:
##LISTO PROFE MUCHAS GRACIAS##